# Auto-Caption Quick Start Guide

This notebook provides a quick introduction to using Auto-Caption for generating AI-powered captions from videos.

## Installation

First, make sure you have installed the auto-caption package:

```bash
cd /path/to/auto-caption
pip install -e .
```

## 1. Basic Setup

In [2]:
# Import required modules
import os
import sys
from pathlib import Path

# Add the src directory to Python path if running from notebooks
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    src_path = notebook_dir.parent / 'src'
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

from auto_caption.caption_generator import CaptionGenerator
from auto_caption.models import WhisperModel, is_model_downloaded

# Download the Whisper model (only needs to be done once)
model_name = "base"  # Options: tiny, base, small, medium, large

if is_model_downloaded(model_name):
    print(f"Whisper '{model_name}' model already downloaded!")
else:
    print(f"Downloading Whisper '{model_name}' model...")
    whisper_model = WhisperModel(model_name)
    whisper_model.download()
    print("Model downloaded successfully!")

Whisper 'base' model already downloaded!


## 2. Generate Basic Captions

In [7]:
# Initialize caption generator
caption_gen = CaptionGenerator(
    model_name="base",
    language="en",  # Set to None for auto-detection
    verbose=True
)

# Set your video path
video_path = "/home/nst/PythonProjects/auto-caption/output.mp4"  # Replace with your video file
output_dir = "output/captions"
os.makedirs(output_dir, exist_ok=True)

# Generate captions
result = caption_gen.generate(
    video_path=video_path,
    temperature=0.0,  # Lower temperature for more deterministic results
    word_timestamps=True  # Enable word-level timing
)

# Save the output in different formats
formats = ["srt", "vtt", "txt", "json"]
for fmt in formats:
    output_path = os.path.join(output_dir, f"captions.{fmt}")
    caption_gen.save_output(result, output_path, fmt)
    print(f"✅ Saved {fmt.upper()} format to: {output_path}")

print(f"\n📁 All outputs saved to: {output_dir}")
print(f"🔤 Detected language: {result['language']}")
print(f"📝 Total segments: {len(result['segments'])}")

Loading Whisper model: base
Model loaded successfully on device: cpu
Analyzing video for smart caption positioning...


I0000 00:00:1751390519.025321  524647 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751390519.029528  525876 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1751390519.047626  524647 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751390519.049725  525888 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
100%|████████████████████████████████████████████████████████████████████| 21.5M/21.5M [00:09<00:00, 2.27MB/s]
I0000 00:00:1751390531.011779  524647 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751390531.013363  526050 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)


Analyzed 31 frames for object detection
Extracting audio from: /home/nst/PythonProjects/auto-caption/output.mp4
Transcribing audio...
[00:00.000 --> 00:05.060]  I just want to put something into perspective really quick that when men built the worlds,
[00:05.640 --> 00:10.560]  we needed the worlds to be built. The world was nothing before men built it.
[00:11.280 --> 00:14.940]  And the reason men even built the world was for a safe
✅ Saved SRT format to: output/captions/captions.srt
✅ Saved VTT format to: output/captions/captions.vtt
✅ Saved TXT format to: output/captions/captions.txt
✅ Saved JSON format to: output/captions/captions.json

📁 All outputs saved to: output/captions
🔤 Detected language: en
📝 Total segments: 3


In [8]:
# View the transcription
print("\n=== Full Transcription ===")
print(result['text'][:500] + "..." if len(result['text']) > 500 else result['text'])

print("\n=== First 3 Caption Segments ===")
for i, segment in enumerate(result['segments'][:3]):
    print(f"{i+1}. [{segment['start']:.1f}s - {segment['end']:.1f}s] {segment['text']}")


=== Full Transcription ===
 I just want to put something into perspective really quick that when men built the worlds, we needed the worlds to be built. The world was nothing before men built it. And the reason men even built the world was for a safe

=== First 3 Caption Segments ===
1. [0.0s - 5.1s] I just want to put something into perspective really quick that when men built the worlds
2. [5.6s - 10.6s] we needed the worlds to be built. The world was nothing before men built it
3. [11.3s - 14.9s] And the reason men even built the world was for a safe


## 3. Generate Emotion-Aware Captions

In [5]:
import torch
from auto_caption.emotion_detector import EmotionDetector
from auto_caption.caption_styler import CaptionStyler, Platform, StyleIntensity

# Set your video path
video_path = "/home/nst/PythonProjects/auto-caption/output.mp4"  # Replace with your video file

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize components for emotion-aware captions
emotion_detector = EmotionDetector(device=device)
caption_styler = CaptionStyler(
    default_platform=Platform.TIKTOK,  # Options: TIKTOK, INSTAGRAM, YOUTUBE_SHORTS
    default_intensity=StyleIntensity.MEDIUM  # Options: SUBTLE, MEDIUM, INTENSE
)

# Create emotion-aware caption generator
emotion_caption_gen = CaptionGenerator(
    model_name="base",
    enable_emotion_detection=True,
    emotion_detector=emotion_detector,
    caption_styler=caption_styler,
    verbose=True
)

# Generate emotion-aware captions
output_dir_emotion = "output/emotion_captions"
os.makedirs(output_dir_emotion, exist_ok=True)

emotion_result = emotion_caption_gen.generate(
    video_path=video_path,
    detect_emotions=True,
    style_captions=True,
    word_timestamps=True,
    enable_smart_positioning=True
)

# Save styled captions
emotion_caption_gen.save_output(emotion_result, os.path.join(output_dir_emotion, "captions.srt"), "srt")
emotion_caption_gen.save_output(emotion_result, os.path.join(output_dir_emotion, "captions.ass"), "ass")

print("\n✅ Emotion-aware captions generated!")
print("🎭 Emotions detected and styled appropriately")

Using device: cpu


Some weights of the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition were not used when initializing Wav2Vec2ForSequenceClassification: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'classifier.output.bias', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v', 'classifier.output.weight', 'classifier.dense.bias', 'classifier.dense.weight']
- This IS expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at ehcalabres/wav2vec2-lg-xlsr-e

Loading Whisper model: base
Model loaded successfully on device: cpu
Analyzing video for smart caption positioning...


I0000 00:00:1751429517.285648   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751429517.287533   50761 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
I0000 00:00:1751429517.293515   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751429517.294727   50772 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
I0000 00:00:1751429517.370607   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751429517.371690   50783 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)


Analyzed 31 frames for object detection
Extracting audio from: /home/nst/PythonProjects/auto-caption/output.mp4
Transcribing audio...
Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:05.060]  I just want to put something into perspective really quick that when men built the worlds,
[00:05.640 --> 00:10.560]  we needed the worlds to be built. The world was nothing before men built it.
[00:11.280 --> 00:14.940]  And the reason men even built the world was for a safe
Generated ASS file: output/emotion_captions/captions.ass

✅ Emotion-aware captions generated!
🎭 Emotions detected and styled appropriately


In [7]:
# Display detected emotions if available
if 'emotion_results' in emotion_result:
    print("\n=== Detected Emotions ===")
    for i, emotion in enumerate(emotion_result['emotion_results'][:5]):
        print(f"{i+1}. [{emotion['start']:.1f}s - {emotion['end']:.1f}s] {emotion['emotion']} (confidence: {emotion['confidence']:.2f})")

## 4. Word-by-Word Captions

Create engaging word-by-word captions that display text synchronized with speech:

In [10]:
# Import required modules for word-by-word processing
from auto_caption.word_timing import WordTimingProcessor, WordAnimationStyle
from auto_caption.subtitle import ASSGenerator

# Create standard caption generator
word_caption_gen = CaptionGenerator(
    model_name="base",
    verbose=True
)

# Generate captions with word timestamps
output_dir_word = "output/word_by_word_captions"
os.makedirs(output_dir_word, exist_ok=True)

# Generate captions with word-level timestamps
result = word_caption_gen.generate(
    video_path=video_path,
    word_timestamps=True,  # Enable word-level timestamps from Whisper
    temperature=0.0
)

# Process into word-by-word timing
word_processor = WordTimingProcessor(
    animation_style=WordAnimationStyle.POP_IN,  # Options: TYPEWRITER, FADE_IN, POP_IN, SLIDE_IN, BOUNCE_IN, WAVE, KARAOKE, EMPHASIS
    words_per_second=3.0,  # Average reading speed
    min_word_duration=0.15,  # Minimum time per word
    max_word_duration=0.8,  # Maximum time per word
)

# Process segments for word-by-word display
word_segments = word_processor.process_segments(result['segments'])

# Create caption data for word-by-word display
word_result = result.copy()
word_result['word_segments'] = []
for segment in word_segments:
    word_result['word_segments'].append({
        'segment_index': segment.segment_index,
        'words': [{
            'word': w.word,
            'start_time': w.start_time,
            'end_time': w.end_time,
            'duration': w.duration,
            'is_emphasized': w.is_emphasized
        } for w in segment.words]
    })
word_result['word_by_word'] = True
word_result['word_animation'] = WordAnimationStyle.POP_IN.value

# Save word-by-word captions
word_caption_gen.save_output(word_result, os.path.join(output_dir_word, "captions_word.json"), "json")

# Generate ASS file with word-by-word styling
ass_gen = ASSGenerator()
ass_file = os.path.join(output_dir_word, "captions_word.ass")
ass_gen.generate_ass_file(word_result, ass_file, "Word-by-Word Captions")

print("\n✅ Word-by-word captions generated!")
print(f"📝 Animation style: {WordAnimationStyle.POP_IN.value}")
print(f"⏱️ Words per second: 3.0")
print(f"📁 Files saved to: {output_dir_word}")

Loading Whisper model: base
Model loaded successfully on device: cpu
Analyzing video for smart caption positioning...


I0000 00:00:1751430082.696870   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751430082.697972   95452 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
I0000 00:00:1751430082.706380   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751430082.707616   95463 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
I0000 00:00:1751430082.812370   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751430082.818208   95476 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)


Analyzed 31 frames for object detection
Extracting audio from: /home/nst/PythonProjects/auto-caption/output.mp4
Transcribing audio...
Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:05.060]  I just want to put something into perspective really quick that when men built the worlds,
[00:05.640 --> 00:10.560]  we needed the worlds to be built. The world was nothing before men built it.
[00:11.280 --> 00:14.940]  And the reason men even built the world was for a safe

✅ Word-by-word captions generated!
📝 Animation style: pop_in
⏱️ Words per second: 3.0
📁 Files saved to: output/word_by_word_captions


In [11]:
# View word timing details
if 'word_segments' in word_result:
    print("\n=== Word Timing Details (First Segment) ===")
    first_segment = word_result['word_segments'][0]
    for i, word_info in enumerate(first_segment['words'][:10]):  # Show first 10 words
        print(f"{i+1}. '{word_info['word']}' [{word_info['start_time']:.2f}s - {word_info['end_time']:.2f}s] (duration: {word_info['duration']:.2f}s)")
        if word_info.get('is_emphasized'):
            print(f"   ⭐ Emphasized word!")


=== Word Timing Details (First Segment) ===
1. 'I' [0.00s - 0.24s] (duration: 0.24s)
2. 'just' [0.24s - 0.36s] (duration: 0.12s)
3. 'want' [0.36s - 0.54s] (duration: 0.18s)
4. 'to' [0.54s - 0.62s] (duration: 0.08s)
5. 'put' [0.62s - 0.74s] (duration: 0.12s)
6. 'something' [0.74s - 1.10s] (duration: 0.36s)
7. 'into' [1.10s - 1.28s] (duration: 0.18s)
8. 'perspective' [1.28s - 1.78s] (duration: 0.50s)
9. 'really' [1.78s - 2.12s] (duration: 0.34s)
10. 'quick' [2.12s - 2.54s] (duration: 0.42s)


### Word-by-Word with Emotion Detection

Combine word-by-word captions with emotion detection for maximum engagement:

In [14]:
# Create caption generator with emotion detection
word_emotion_caption_gen = CaptionGenerator(
    model_name="base",
    enable_emotion_detection=True,
    emotion_detector=emotion_detector,
    caption_styler=caption_styler,
    verbose=True
)

# Generate emotion-aware captions with word timestamps
output_dir_word_emotion = "output/word_emotion_captions"
os.makedirs(output_dir_word_emotion, exist_ok=True)

emotion_word_result = word_emotion_caption_gen.generate(
    video_path=video_path,
    word_timestamps=True,
    detect_emotions=True,
    style_captions=True,
    enable_smart_positioning=True
)

# Process with word timing, emotion awareness enabled
emotion_word_processor = WordTimingProcessor(
    animation_style=WordAnimationStyle.EMPHASIS,
    words_per_second=3.0
)

# Process segments with emotion data
emotion_word_segments = emotion_word_processor.process_segments(emotion_word_result['segments'])

# Update result with word segments
emotion_word_result['word_segments'] = []
for segment in emotion_word_segments:
    emotion_word_result['word_segments'].append({
        'segment_index': segment.segment_index,
        'words': [{
            'word': w.word,
            'start_time': w.start_time,
            'end_time': w.end_time,
            'duration': w.duration,
            'is_emphasized': w.is_emphasized,
            'emotion': w.emotion.value if w.emotion else None
        } for w in segment.words]
    })
emotion_word_result['word_by_word'] = True
emotion_word_result['word_animation'] = WordAnimationStyle.EMPHASIS.value

# Generate ASS file with emotion-aware word-by-word styling
ass_file = os.path.join(output_dir_word_emotion, "captions.ass")
ass_gen.generate_ass_file(emotion_word_result, ass_file, "Emotion Word-by-Word")

print("\n✅ Emotion-aware word-by-word captions generated!")
print("🎭 Words styled based on detected emotions")
print("✨ Key words emphasized automatically")

Loading Whisper model: base
Model loaded successfully on device: cpu
Analyzing video for smart caption positioning...


I0000 00:00:1751430611.582679   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751430611.583828  191063 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
I0000 00:00:1751430611.589967   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751430611.592409  191074 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
I0000 00:00:1751430611.662038   11565 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751430611.663040  191086 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) UHD Graphics (CML GT2)


Analyzed 31 frames for object detection
Extracting audio from: /home/nst/PythonProjects/auto-caption/output.mp4
Transcribing audio...
Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:05.060]  I just want to put something into perspective really quick that when men built the worlds,
[00:05.640 --> 00:10.560]  we needed the worlds to be built. The world was nothing before men built it.
[00:11.280 --> 00:14.940]  And the reason men even built the world was for a safe

✅ Emotion-aware word-by-word captions generated!
🎭 Words styled based on detected emotions
✨ Key words emphasized automatically


### Different Animation Styles

Try different word animation styles for various content types:

In [ ]:
# Demonstrate different animation styles
animation_styles = [
    (WordAnimationStyle.TYPEWRITER, "Natural sequential appearance"),
    (WordAnimationStyle.FADE_IN, "Smooth opacity transition"),
    (WordAnimationStyle.POP_IN, "Scale with bounce effect"),
    (WordAnimationStyle.KARAOKE, "Highlight as spoken"),
    (WordAnimationStyle.EMPHASIS, "Key words emphasized")
]

print("Available Word Animation Styles:")
for style, description in animation_styles:
    print(f"\n{style.value.upper()}:")
    print(f"  - {description}")
    
    # Create processor with this style
    style_processor = WordTimingProcessor(
        animation_style=style,
        words_per_second=3.0
    )
    
    # Example usage
    print(f"  - Command: auto-caption generate video.mp4 --word-by-word --word-animation {style.value}")

## 5. Combine All Features

Here's how to combine word-by-word captions with emotion detection and smart positioning:

In [ ]:
# Create the ultimate caption generator with all features
full_featured_gen = CaptionGenerator(
    model_name="base",
    enable_emotion_detection=True,
    emotion_detector=emotion_detector,
    caption_styler=CaptionStyler(
        default_platform=Platform.TIKTOK,
        default_intensity=StyleIntensity.INTENSE
    ),
    verbose=True
)

# Generate captions with all features
output_dir_full = "output/full_featured_captions"
os.makedirs(output_dir_full, exist_ok=True)

full_result = full_featured_gen.generate(
    video_path=video_path,
    word_timestamps=True,
    detect_emotions=True,
    style_captions=True,
    enable_smart_positioning=True,
    temperature=0.0
)

# Process with advanced word timing
advanced_processor = WordTimingProcessor(
    animation_style=WordAnimationStyle.EMPHASIS,
    words_per_second=3.0,
    enable_emotion_emphasis=True,
    emphasis_duration_multiplier=1.5
)

# Process segments
full_word_segments = advanced_processor.process_segments(
    full_result['segments'],
    emotion_results=full_result.get('emotion_results')
)

# Add word segments to result
full_result['word_segments'] = []
for segment in full_word_segments:
    full_result['word_segments'].append({
        'segment_index': segment.segment_index,
        'words': [{
            'word': w.word,
            'start_time': w.start_time,
            'end_time': w.end_time,
            'duration': w.duration,
            'is_emphasized': w.is_emphasized,
            'emotion': w.emotion.value if w.emotion else None
        } for w in segment.words]
    })
full_result['word_by_word'] = True
full_result['word_animation'] = WordAnimationStyle.EMPHASIS.value

# Save in multiple formats
full_featured_gen.save_output(full_result, os.path.join(output_dir_full, "captions.json"), "json")

# Generate ASS with all features
ass_file = os.path.join(output_dir_full, "captions.ass")
ass_gen.generate_ass_file(full_result, ass_file, "Full Featured Captions")

print("\n🎯 Full-featured captions generated!")
print("✅ Word-by-word timing")
print("✅ Emotion detection and styling")
print("✅ Smart positioning")
print("✅ Platform-optimized styling")

## 6. Batch Processing Multiple Videos

Process multiple videos with the same settings:

In [ ]:
from pathlib import Path

def process_video_folder(folder_path, output_base="output/batch"):
    """Process all videos in a folder."""
    
    # Find all video files
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv']
    video_files = []
    for ext in video_extensions:
        video_files.extend(Path(folder_path).glob(f'*{ext}'))
    
    if not video_files:
        print(f"No video files found in {folder_path}")
        return
    
    print(f"Found {len(video_files)} videos to process:\n")
    
    # Initialize caption generator once
    caption_gen = CaptionGenerator(model_name="base")
    
    # Process each video
    for i, video_path in enumerate(video_files, 1):
        print(f"[{i}/{len(video_files)}] Processing: {video_path.name}")
        
        output_dir = os.path.join(output_base, video_path.stem)
        os.makedirs(output_dir, exist_ok=True)
        
        try:
            # Generate captions
            result = caption_gen.generate(
                video_path=str(video_path),
                temperature=0.0
            )
            
            # Save as SRT
            output_path = os.path.join(output_dir, "captions.srt")
            caption_gen.save_output(result, output_path, "srt")
            
            print(f"   ✅ Success! Language: {result['language']}, Segments: {len(result['segments'])}")
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")
    
    print(f"\n🎉 Batch processing complete! Results saved to: {output_base}")

# Example usage (uncomment and set your folder path)
# video_folder = "path/to/your/videos"
# process_video_folder(video_folder)

## 4. Analyze Video Emotions Only

In [ ]:
# Just analyze emotions without generating captions
from auto_caption.emotion_detector import EmotionDetector
import json

def analyze_video_emotions(video_path):
    """Analyze emotions in a video without generating captions."""
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    detector = EmotionDetector(device=device)
    
    print(f"Analyzing video emotions (device: {device})...")
    
    # Detect emotions
    emotion_result = detector.detect_emotions(video_path)
    
    # Save results
    output_path = "output/emotion_analysis.json"
    os.makedirs("output", exist_ok=True)
    
    # Convert to dictionary for JSON serialization
    results_dict = {
        "dominant_emotion": emotion_result.dominant_emotion.value,
        "average_confidence": emotion_result.average_confidence,
        "emotion_distribution": emotion_result.emotion_distribution,
        "temporal_emotions": emotion_result.temporal_emotions,
        "metadata": emotion_result.metadata
    }
    
    with open(output_path, 'w') as f:
        json.dump(results_dict, f, indent=2)
    
    # Display summary
    print(f"\n📊 Emotion Analysis Summary:")
    print(f"• Dominant emotion: {emotion_result.dominant_emotion.value}")
    print(f"• Average confidence: {emotion_result.average_confidence:.2f}")
    print(f"• Total segments analyzed: {len(emotion_result.temporal_emotions)}")
    
    # Show emotion distribution
    print("\n📈 Emotion Distribution:")
    for emotion, score in sorted(emotion_result.emotion_distribution.items(), 
                                key=lambda x: x[1], reverse=True):
        print(f"• {emotion}: {score:.2%}")
    
    print(f"\n💾 Full results saved to: {output_path}")
    
    return emotion_result

# Example usage
# emotions = analyze_video_emotions("path/to/video.mp4")

## 5. Advanced Caption Styling

In [ ]:
# Example of generating captions with specific platform styling
from auto_caption.caption_styler import Platform, StyleIntensity

# Platform-specific generators
platforms = {
    "tiktok": (Platform.TIKTOK, StyleIntensity.INTENSE),
    "instagram": (Platform.INSTAGRAM, StyleIntensity.MEDIUM),
    "youtube": (Platform.YOUTUBE_SHORTS, StyleIntensity.SUBTLE)
}

for platform_name, (platform, intensity) in platforms.items():
    print(f"\n🎨 Generating {platform_name.upper()} styled captions...")
    
    # Create platform-specific styler
    styler = CaptionStyler(
        default_platform=platform,
        default_intensity=intensity
    )
    
    # Create generator with specific styling
    platform_gen = CaptionGenerator(
        model_name="base",
        caption_styler=styler,
        verbose=False
    )
    
    # Generate and save
    result = platform_gen.generate(video_path, style_captions=True)
    
    output_dir = f"output/{platform_name}_style"
    os.makedirs(output_dir, exist_ok=True)
    
    # Save ASS format for styled subtitles
    output_path = os.path.join(output_dir, "captions.ass")
    platform_gen.save_output(result, output_path, "ass")
    
    print(f"   ✅ Saved to: {output_path}")